# Detector comparison — six detectors on a shared SNR axis

Compares **all six** signal detectors on the *same* frames, ground truth, and
metrics, plotted against a physically-meaningful **SNR (dB)** axis (shared
-20..+40 dB range):

| detector | what it is | source |
|---|---|---|
| `coherent_power` | deployed coherent power detector | C++ offline binary |
| `cuda_dino` | zero-shot DINOv3 | C++ offline binary |
| `3dB_power` | moving-average power baseline (non-ML) | `run_baseline_offline.py` |
| `blob_detection` | image-processing blobs baseline (non-ML) | `run_baseline_offline.py` |
| `yolo` | fine-tuned YOLO26 | `run_ml_detectors_offline.py` |
| `dino_finetuned` | fine-tuned DINOv3 (M2_ft) | `run_ml_detectors_offline.py` |

Run the whole sweep headless with `run_full_comparison.py` (see `README.md`); it
lands every detector as a sibling dir in one batch root and writes the serialized
`snr_results` object + `snr_plots/`. This notebook is the visual review — reusing
the shared helpers (`eval_viz`, `mask_eval_metrics`) plus the SNR-axis plotter
`plot_snr_results` (in this folder). Every detector is scored by the *identical*
`eval_detector_masks.py` code path, so the comparison is fair.

* **Panels** — spectrogram + ground-truth boxes, then each detector's mask overlaid.
* **Aggregate (SNR)** — detection rate + frame pixel metrics vs SNR, all six detectors.


In [1]:
# %load_ext autoreload
# %autoreload 2
import sys
from pathlib import Path

# The shared eval helpers live in the sibling signal_detection_experiments/ folder.
_HELPERS = (Path.cwd().parent if Path.cwd().name == 'baseline_comparisons' else Path.cwd()) / 'signal_detection_experiments'
if str(_HELPERS) not in sys.path:
    sys.path.insert(0, str(_HELPERS))

import eval_viz as v
import mask_eval_metrics as mem

# --- parameters ---
# BATCH_ROOT is the batch root that holds BOTH the trained detector dirs AND the
# baseline dirs (3dB_power/, blob_detection/) written by run_baseline_offline.py.
# It may be a batch root, a detector dir, or a single run dir — resolve_layout() copes.
# This combined root holds the baselines + a symlink to the trained coherent_power/
# run, all sharing the same ground truth. batch_runs/ lives under the sibling
# signal_detection_experiments/ folder.
BATCH_ROOT = Path('/home/sat3737/holohub-dev/applications/usrp_wideband_signal_detection/infocom_evals/signal_detection_experiments/batch_runs/3db_blob_coherent_yolo_dino_all6_0_60dBFiles')
RESULTS = BATCH_ROOT / 'snr_results'

CAPTURE_DIRS = [Path('/home/bqn82/captures'),
                Path('../../generated_inputs').resolve()]   # for spectrogram reconstruction
FILE_STEM = 'attenuation_dB_35'   # disambiguates when several captures are present
DETECTORS = None                  # None = auto-detect all detectors present (trained + baseline)
FRAME = 100                       # pick a frame with signals (early frames may be empty)

layout = v.resolve_layout(BATCH_ROOT, FILE_STEM)
print('resolved layout:', layout)

resolved layout: {'batch_root': PosixPath('/home/sat3737/holohub-dev/applications/usrp_wideband_signal_detection/infocom_evals/signal_detection_experiments/batch_runs/3db_blob_coherent_yolo_dino_all6_0_60dBFiles'), 'file_stem': 'attenuation_dB_35', 'detectors': ['3dB_power', 'blob_detection', 'coherent_power', 'cuda_dino', 'dino_finetuned', 'yolo']}


## What is available?

In [2]:
file_stem = layout['file_stem']
detectors_present = layout['detectors']
print('file_stem:', file_stem, '| detectors:', detectors_present)
if detectors_present:
    frames = v.available_frames(layout['batch_root'], detectors_present[0], file_stem)
    print(f'{len(frames)} frames available, e.g.', frames[:10])

file_stem: attenuation_dB_35 | detectors: ['3dB_power', 'blob_detection', 'coherent_power', 'cuda_dino', 'dino_finetuned', 'yolo']
334 frames available, e.g. [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]


## Render the comparison panels for one frame

`[ GT | coherent_power | cuda_dino | 3dB_power | blob_detection ]` — whichever are present.

In [3]:
bundle = v.load_frame_bundle_smart(BATCH_ROOT, FRAME, file_stem=FILE_STEM,
                                   detectors=DETECTORS, capture_dirs=CAPTURE_DIRS)
print('grid', bundle.fft_rows, 'x', bundle.fft_cols,
      '| span', bundle.span_hz/1e6, 'MHz',
      '| detectors', list(bundle.detector_masks),
      '| GT regions', len(bundle.gt_items))
fig = v.plot_frame_panels(bundle)
fig

FileNotFoundError: GT mask missing for frame 100 in /home/sat3737/holohub-dev/applications/usrp_wideband_signal_detection/infocom_evals/signal_detection_experiments/batch_runs/3db_blob_coherent_yolo_dino_all6_0_60dBFiles/3dB_power/attenuation_dB_35

## Review a random sample of frames (annotated + noise-only)

A reproducible random draw of annotated frames + one noise-only frame. Watch the
baselines' behavior on the noise-only frame — a naive `3dB_power` detector tends to
speckle on noise (tune `min_run_bins`), while `blob_detection`'s area filter suppresses it.

In [ ]:
from IPython.display import display

SAMPLE = v.sample_review_frames(BATCH_ROOT, FILE_STEM, n_annotated=5, n_noise=1, seed=7)
print(f"{SAMPLE['annotated_available']} annotated / {SAMPLE['noise_available']} noise-only frames available")
print('reviewing frames:', SAMPLE['review_frames'], ' (noise-only:', SAMPLE['noise_frames'], ')')

for fr in SAMPLE['review_frames']:
    b = v.load_frame_bundle_smart(BATCH_ROOT, fr, file_stem=FILE_STEM,
                                  detectors=DETECTORS, capture_dirs=CAPTURE_DIRS)
    tag = 'NOISE-ONLY' if fr in SAMPLE['noise_frames'] else f'{len(b.gt_items)} GT regions'
    print(f'--- frame {fr}: {tag} | detectors {list(b.detector_masks)} ---')
    display(v.plot_frame_panels(b))

## (optional) Per-frame metrics for context

Pixel precision/recall/F1/IoU for the same frame, per detector. GT masks are *filled
bounding boxes*, so sparse detectors show high precision / low recall — lean on region
coverage for 'did it find the signal'.

In [ ]:
import math
src_meta = None
for d in CAPTURE_DIRS:
    cand = Path(d) / f'{file_stem}.sigmf-meta'
    if cand.exists():
        src_meta = cand; break
for det in (DETECTORS or detectors_present):
    run_dir = layout['batch_root'] / det / file_stem
    fr, rr = mem.evaluate_run(run_dir, det, file_stem, sigmf_meta_path=src_meta, frame_limit=FRAME)
    row = next((r for r in fr if r['frame_number'] == FRAME), None)
    if row:
        flag = '' if row.get('mask_present', True) else '  [NO MASK for this frame]'
        print(f"{det:16s}: P={row['precision']:.3f} R={row['recall']:.3f} "
              f"F1={row['f1']:.3f} IoU={row['iou']:.3f} FParea={row['fp_area_fraction']:.4f}{flag}")

## Aggregate performance plots (whole sweep)

Comparison figures from the tidy fact tables (`eval_detector_masks.py` output):
detection rate **vs power (SNR)** faceted by signal class / bandwidth / pulse length;
performance vs bandwidth and vs duration; and frame-level precision / recall / F1 /
pixel-IoU + false-positive area vs power. The baselines get their own color/marker
(`3dB_power` green ^, `blob_detection` purple D) via `plot_eval_results.DETECTOR_STYLE`.

Higher attenuation (dB) = lower SNR, so these curves *are* the performance-vs-SNR view.
Batch CLI equivalent: `python3 ../signal_detection_experiments/plot_eval_results.py --tables-dir <dir> --det-threshold 0.1`.

In [ ]:
import importlib, plot_eval_results as pe
importlib.reload(pe)
from IPython.display import display

# Point at the eval tables (eval_detector_masks.py --out-dir). Usually == BATCH_ROOT.
TABLES_DIR    = BATCH_ROOT
DET_THRESHOLD = 0.1

region = pe.load_region(TABLES_DIR / 'region_metrics.csv')
frame  = pe.load_frame(TABLES_DIR / 'frame_pixel_metrics.csv')
print(f'{len(region)} region rows, {len(frame)} frame rows | detectors:',
      sorted({r['detector'] for r in frame}))

# --- detection rate vs POWER (SNR), faceted by signal type / bandwidth / pulse length ---
display(pe.fig_rate_vs_power_by(region, 'signal_class', None, DET_THRESHOLD,
                                'Detection rate vs power, by signal class'))
display(pe.fig_rate_vs_power_by(region, 'bandwidth', pe.BW_ORDER, DET_THRESHOLD,
                                'Detection rate vs power, by bandwidth'))
display(pe.fig_rate_vs_power_by(region, 'pulse_length', pe.LEN_ORDER, DET_THRESHOLD,
                                'Detection rate vs power, by pulse length (time duration)'))

# --- performance (detection rate / IoU / coverage) vs BANDWIDTH and vs DURATION ---
display(pe.fig_metric_vs_bucket(region, 'bandwidth', pe.BW_ORDER, DET_THRESHOLD,
                                'Performance vs signal bandwidth'))
display(pe.fig_metric_vs_bucket(region, 'pulse_length', pe.LEN_ORDER, DET_THRESHOLD,
                                'Performance vs pulse length (time duration)'))

# --- frame-level accuracy + FALSE-POSITIVE area vs POWER ---
display(pe.fig_frame_metrics_vs_power(frame))

## Aggregate performance vs SNR — all six detectors

The physically-meaningful view: detection rate and frame-level pixel metrics vs
**SNR (dB)** rather than attenuation, on a shared **-20..+40 dB** axis so every panel
lines up. Loaded from the serialized `SnrResults` object (`build_snr_results.py`), so
re-styling / re-binning needs no recompute of the captures or fact tables. Consistent
colors/markers per detector (`plot_snr_results.DETECTOR_STYLE`): coherent_power (blue o),
cuda_dino (red s), 3dB_power (green ^), blob_detection (purple D), yolo (orange v),
dino_finetuned (brown P).

Batch CLI equivalent: `python3 plot_snr_results.py --results <BATCH_ROOT>/snr_results --snr-range -20 40`.


In [ ]:
import importlib, snr_measurement as sm, plot_snr_results as ps
importlib.reload(sm); importlib.reload(ps)
from IPython.display import display

# RESULTS = BATCH_ROOT/'snr_results' (base path written by build_snr_results.py).
results = sm.SnrResults.load(RESULTS)
print(f"{len(results.region.get('snr_db', []))} region rows, "
      f"{len(results.frame.get('frame_snr_db', []))} frame rows | detectors:",
      ps.order_detectors({str(d) for d in results.region['detector']}))

SNR_RANGE     = (-20.0, 40.0)   # shared x-axis (matches build/plot defaults)
SNR_BIN_WIDTH = 5.0             # dB (== physical attenuator step)
DET_THRESHOLD = 0.1             # region 'detected' if box coverage >= this

figs = ps.make_all_figures(results, threshold=DET_THRESHOLD,
                           snr_bin_width=SNR_BIN_WIDTH, snr_range=SNR_RANGE)
for name in ('rate_vs_snr_overall', 'rate_vs_snr_by_class',
             'rate_vs_snr_by_bandwidth', 'frame_metrics_vs_snr'):
    print(f'--- {name} ---')
    display(figs[name])


## (optional) Pick example frames by signal class / bandwidth / duration

In [ ]:
import importlib
importlib.reload(v)
from IPython.display import display

region_rows = v.load_region_table(TABLES_DIR / 'region_metrics.csv')

PANEL_FILE_STEM = FILE_STEM   # any capture stem present in the sweep
SIGNAL_CLASS = None           # e.g. 'Narrowband_FM','QPSK','OFDM',... or 'noise' for blank frames
BW           = None           # '<2MHz','2-10MHz','10-25MHz','25-60MHz','>=60MHz'
DURATION     = None           # '<10k','10k-100k','100k-1M','1M-5M','>=5M'
N_EXAMPLES   = 3

opts = v.filter_options(region_rows, PANEL_FILE_STEM)
print(f"available for {PANEL_FILE_STEM} ({opts['n_annotation_rows']} annotations):")
for key in ('signal_class', 'bandwidth', 'pulse_length'):
    print(f'  {key:13s}: {opts[key]}')

frames = v.select_frames(BATCH_ROOT, region_rows, PANEL_FILE_STEM,
                         signal_class=SIGNAL_CLASS, bandwidth=BW, pulse_length=DURATION)
picks = v.pick_spread(frames, N_EXAMPLES)
print(f'\nfilter: class={SIGNAL_CLASS} bw={BW} duration={DURATION} -> {len(frames)} frames; showing {picks}')
for fr in picks:
    b = v.load_frame_bundle_smart(BATCH_ROOT, fr, file_stem=PANEL_FILE_STEM, capture_dirs=CAPTURE_DIRS)
    print(f'--- frame {fr}: GT regions {len(b.gt_items)} ---')
    display(v.plot_frame_panels(b))
if not picks:
    print('No frames match those filters — widen them (set some to None) or check the menu above.')